In [19]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading panels...")
acs   = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_poverty_panel.csv"),
                    dtype={"FIPS": str})
noaa  = pd.read_csv(os.path.join(OUTPUT_DIR, "noaa_heat_panel.csv"),
                    dtype={"FIPS": str})
eia   = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_county_panel.csv"),
                    dtype={"FIPS": str})

# Pad FIPS
for df in [acs, noaa, eia]:
    df["FIPS"] = df["FIPS"].str.zfill(5)

print(f"ACS:  {len(acs)} rows, {acs['FIPS'].nunique()} counties, "
      f"years {acs['year'].min()}–{acs['year'].max()}")
print(f"NOAA: {len(noaa)} rows, {noaa['FIPS'].nunique()} counties, "
      f"years {noaa['year'].min()}–{noaa['year'].max()}")
print(f"EIA:  {len(eia)} rows, {eia['FIPS'].nunique()} counties, "
      f"years {eia['year'].min()}–{eia['year'].max()}")

# ── STEP 1: COLLAPSE NOAA TO ANNUAL SUMMER HEAT ───────────────────────────────
# Sum heat days across June, July, August for each county-year
summer_heat = (noaa[noaa["month"].isin([6, 7, 8])]
               .groupby(["FIPS", "year"])
               .agg(
                   summer_days_90f  = ("days_above_90f",  "sum"),
                   summer_days_95f  = ("days_above_95f",  "sum"),
                   summer_days_100f = ("days_above_100f", "sum"),
               )
               .reset_index())

# Also keep monthly data for the full regression spec
# (regression will use month-level heat bins merged differently)
# For the master panel we keep annual summer totals
print(f"\nSummer heat panel: {len(summer_heat)} county-year rows")

# ── STEP 2: RESTRICT TO OVERLAPPING YEARS ────────────────────────────────────
# ACS: 2012-2024, NOAA: 2012-2024, EIA: 2015-2024
# Overlap: 2015-2024
YEAR_MIN = 2015
YEAR_MAX = 2024

acs_sub        = acs[acs["year"].between(YEAR_MIN, YEAR_MAX)].copy()
summer_heat_sub = summer_heat[summer_heat["year"].between(YEAR_MIN, YEAR_MAX)].copy()
eia_sub        = eia[eia["year"].between(YEAR_MIN, YEAR_MAX)].copy()

# ── STEP 3: MERGE ─────────────────────────────────────────────────────────────
# Start with EIA as base (our outcome variable)
panel = eia_sub.merge(
    acs_sub[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
    on=["FIPS", "year"],
    how="left"
)

panel = panel.merge(
    summer_heat_sub,
    on=["FIPS", "year"],
    how="left"
)

print(f"\nMaster panel: {len(panel)} rows")
print(f"Unique counties: {panel['FIPS'].nunique()}")
print(f"Years: {sorted(panel['year'].unique())}")

# ── STEP 4: CONSTRUCT KEY VARIABLES ──────────────────────────────────────────
# Binary high-poverty indicator (above median poverty rate)
median_pov = panel["pct_below_150pct_poverty"].median()
panel["high_poverty"] = (panel["pct_below_150pct_poverty"] >= median_pov).astype(int)

# Demeaned poverty rate (for triple interaction interpretation)
panel["poverty_demeaned"] = (panel["pct_below_150pct_poverty"] - median_pov)

# Log outcome variables (already computed in EIA script)
# log_res_sales_per_customer and log_implied_price already present

# Heat bins following Doremus (8 bins)
# Using days_above_95f as primary heat measure
bin_edges  = [-1, 0, 3, 7, 12, 17, 22, 27, 31]
bin_labels = ["bin1_0days", "bin2_1to3", "bin3_4to7", "bin4_8to12",
              "bin5_13to17", "bin6_18to22", "bin7_23to27", "bin8_28plus"]

panel["heat_bin_95f"] = pd.cut(
    panel["summer_days_95f"],
    bins=bin_edges,
    labels=bin_labels
)

# ── STEP 5: DIAGNOSTICS ───────────────────────────────────────────────────────
print(f"\nMissing values:")
key_vars = ["pct_below_150pct_poverty", "summer_days_95f",
            "log_res_sales_per_customer", "log_implied_price",
            "dominant_ownership_type"]
for v in key_vars:
    n_miss = panel[v].isna().sum()
    print(f"  {v}: {n_miss} missing ({n_miss/len(panel):.1%})")

print(f"\nOwnership type distribution:")
print(panel["dominant_ownership_type"].value_counts())

print(f"\nHeat bin distribution:")
print(panel["heat_bin_95f"].value_counts().sort_index())

print(f"\nMedian poverty rate: {median_pov:.3f}")
print(f"High poverty counties: {panel['high_poverty'].sum()} "
      f"({panel['high_poverty'].mean():.1%})")

# ── STEP 6: SUMMARY STATISTICS TABLE ─────────────────────────────────────────
print(f"\n{'='*60}")
print("TABLE 1 — SUMMARY STATISTICS")
print(f"{'='*60}")

table_vars = {
    "Log res. sales per customer (MWh)": "log_res_sales_per_customer",
    "Log implied price ($/MWh)":         "log_implied_price",
    "Poverty rate (% below 150% FPL)":   "pct_below_150pct_poverty",
    "Summer days above 90°F":            "summer_days_90f",
    "Summer days above 95°F":            "summer_days_95f",
    "Summer days above 100°F":           "summer_days_100f",
    "Residential customers (000s)":      "res_customers_total",
}

rows = []
for label, col in table_vars.items():
    src = panel[col].copy()
    if col == "res_customers_total":
        src = src / 1000
    rows.append({
        "Variable": label,
        "N":    int(src.count()),
        "Mean": src.mean(),
        "SD":   src.std(),
        "P10":  src.quantile(0.10),
        "P50":  src.quantile(0.50),
        "P90":  src.quantile(0.90),
    })

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False, float_format="{:.3f}".format))

# ── STEP 7: SANITY CHECK ──────────────────────────────────────────────────────
print(f"\nAutauga County (01001) master panel:")
print(panel[panel["FIPS"] == "01001"][
    ["FIPS", "year", "pct_below_150pct_poverty",
     "summer_days_95f", "log_res_sales_per_customer",
     "log_implied_price", "dominant_ownership_type"]
].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
panel.to_csv(os.path.join(OUTPUT_DIR, "master_panel.csv"), index=False)
print(f"\nSaved master_panel.csv")
print(f"Final panel: {len(panel)} county-year observations")
print(f"Counties: {panel['FIPS'].nunique()}, Years: {YEAR_MIN}–{YEAR_MAX}")

Loading panels...
ACS:  41869 rows, 3234 counties, years 2012–2024
NOAA: 484692 rows, 3107 counties, years 2012–2024
EIA:  30928 rows, 3115 counties, years 2015–2024

Summer heat panel: 40391 county-year rows

Master panel: 30928 rows
Unique counties: 3115
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values:
  pct_below_150pct_poverty: 25 missing (0.1%)
  summer_days_95f: 218 missing (0.7%)
  log_res_sales_per_customer: 0 missing (0.0%)
  log_implied_price: 0 missing (0.0%)
  dominant_ownership_type: 0 missing (0.0%)

Ownership type distribution:
dominant_ownership_type
IOU            25294
Cooperative     4094
Municipal       1540
Name: count, dtype: int64

Heat bin distribution:
heat_bin_95f
bin1_0days     12218
bin2_1to3       5158
bin3_4to7       3493
bin4_8to12      2373
bin5_13to17     1538
bin6_18to22     1147
bin7_23to27     1007
bin8_28plus      623
Name: count, dtype: int64

Median poverty rate: 0.245
High poverty counties: 15452 (50.0%)

TABLE

In [3]:
import pandas as pd

noaa = pd.read_csv("/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv",
                   dtype={"FIPS": str})
noaa["FIPS"] = noaa["FIPS"].str.zfill(5)

# Check how many counties have all 3 summer months in 2020
summer_2020 = noaa[(noaa["year"] == 2020) & (noaa["month"].isin([6,7,8]))]
months_per_county = summer_2020.groupby("FIPS")["month"].count()

print("Month counts per county in summer 2020:")
print(months_per_county.value_counts())

# How many files do we have total?
import glob
files = glob.glob("/Users/aliabonanno/public_policy_project/tempdata/tmax-*-cty-scaled.csv")
print(f"\nTotal NOAA files: {len(files)}")

# Which months do we have for 2020?
months_2020 = sorted([int(f.split("tmax-2020")[1].split("-")[0]) 
                      for f in files if "tmax-2020" in f])
print(f"Months available for 2020: {months_2020}")

Month counts per county in summer 2020:
month
3    3107
Name: count, dtype: int64

Total NOAA files: 169
Months available for 2020: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [5]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel.csv",
                     dtype={"FIPS": str})
noaa_heat = pd.read_csv("/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv",
                        dtype={"FIPS": str})

master["FIPS"] = master["FIPS"].str.zfill(5)
noaa_heat["FIPS"] = noaa_heat["FIPS"].str.zfill(5)

# Check summer heat collapse
summer = noaa_heat[noaa_heat["month"].isin([6,7,8])]
summer_agg = summer.groupby(["FIPS","year"])["days_above_95f"].sum().reset_index()

print(f"Summer heat rows: {len(summer_agg)}")
print(f"Summer heat years: {sorted(summer_agg['year'].unique())}")

# Check how many EIA FIPS are in summer heat
eia_fips = set(master["FIPS"].unique())
noaa_fips = set(summer_agg["FIPS"].unique())
print(f"\nEIA counties: {len(eia_fips)}")
print(f"NOAA counties: {len(noaa_fips)}")
print(f"In EIA but not NOAA: {len(eia_fips - noaa_fips)}")
print(f"In NOAA but not EIA: {len(noaa_fips - eia_fips)}")

# Check a missing county
missing_fips = list(eia_fips - noaa_fips)[:5]
print(f"\nSample EIA FIPS missing from NOAA: {missing_fips}")
print(f"\nMaster panel missing heat by year:")
print(master.groupby("year")["summer_days_95f"].apply(
    lambda x: x.isna().sum()).reset_index())

Summer heat rows: 40391
Summer heat years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

EIA counties: 3115
NOAA counties: 3107
In EIA but not NOAA: 1525
In NOAA but not EIA: 1517

Sample EIA FIPS missing from NOAA: ['19195', '49017', '26163', '53011', '28089']

Master panel missing heat by year:
   year  summer_days_95f
0  2015             1517
1  2016             1517
2  2017             1518
3  2018             1518
4  2019             1522
5  2020             1506
6  2021             1506
7  2022             1509
8  2023             1510
9  2024             1512


In [11]:
import pandas as pd

noaa = pd.read_csv("/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv",
                   dtype={"FIPS": str})
noaa["FIPS"] = noaa["FIPS"].str.zfill(5)

# Check what FIPS NOAA has for Iowa - full list
iowa_noaa = sorted(noaa[noaa["FIPS"].str.startswith("19")]["FIPS"].unique())
print(f"NOAA Iowa counties: {len(iowa_noaa)}")
print(iowa_noaa)

# Check the crosswalk file that came with NOAA data
import pandas as pd
xwalk = pd.read_csv("/Users/aliabonanno/public_policy_project/tempdata/us-state-codes_ncei-to-fips.csv",
                    dtype=str)
print(f"\nCrosswalk columns: {xwalk.columns.tolist()}")
print(xwalk.head(5).to_string())

NOAA Iowa counties: 14
['19001', '19003', '19005', '19007', '19009', '19011', '19013', '19015', '19017', '19019', '19021', '19023', '19025', '19027']

Crosswalk columns: ['state_name', 'postal_code', 'NCEI_code', 'FIPS_code']
   state_name postal_code NCEI_code FIPS_code
0     Alabama          AL        01        01
1     Arizona          AZ        02        04
2    Arkansas          AR        03        05
3  California          CA        04        06
4    Colorado          CO        05        08


In [14]:
import pandas as pd

xwalk = pd.read_csv("/Users/aliabonanno/public_policy_project/tempdata/us-state-codes_ncei-to-fips.csv",
                    dtype=str)
xwalk["NCEI_code"]  = xwalk["NCEI_code"].str.zfill(2)
xwalk["FIPS_code"]  = xwalk["FIPS_code"].str.zfill(2)

noaa = pd.read_csv("/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv",
                   dtype={"FIPS": str})
noaa["FIPS"] = noaa["FIPS"].str.zfill(5)

# Remap state prefix
noaa["ncei_state"]  = noaa["FIPS"].str[:2]
noaa["county_code"] = noaa["FIPS"].str[2:]
noaa = noaa.merge(xwalk[["NCEI_code","FIPS_code"]],
                  left_on="ncei_state", right_on="NCEI_code", how="left")
noaa["FIPS_census"] = noaa["FIPS_code"] + noaa["county_code"]

print("Sample remapping:")
print(noaa[["FIPS","FIPS_census"]].drop_duplicates().head(15).to_string())
print(f"\nUnique census FIPS after remap: {noaa['FIPS_census'].nunique()}")
print(f"Any unmapped: {noaa['FIPS_census'].isna().sum()}")

Sample remapping:
       FIPS FIPS_census
0     01001       01001
156   01003       01003
312   01005       01005
468   01007       01007
624   01009       01009
780   01011       01011
936   01013       01013
1092  01015       01015
1248  01017       01017
1404  01019       01019
1560  01021       01021
1716  01023       01023
1872  01025       01025
2028  01027       01027
2184  01029       01029

Unique census FIPS after remap: 3107
Any unmapped: 0


In [16]:
print("Iowa remapping (NCEI 13 -> Census 19):")
print(noaa[noaa["ncei_state"] == "13"][["FIPS","FIPS_census"]].drop_duplicates().head(10).to_string())

print("\nMichigan remapping (NCEI 20 -> Census 26):")
print(noaa[noaa["ncei_state"] == "20"][["FIPS","FIPS_census"]].drop_duplicates().head(5).to_string())

Iowa remapping (NCEI 13 -> Census 19):
         FIPS FIPS_census
117624  13001       19001
117780  13003       19003
117936  13005       19005
118092  13007       19007
118248  13009       19009
118404  13011       19011
118560  13013       19013
118716  13015       19015
118872  13017       19017
119028  13019       19019

Michigan remapping (NCEI 20 -> Census 26):
         FIPS FIPS_census
186732  20001       26001
186888  20003       26003
187044  20005       26005
187200  20007       26007
187356  20009       26009


In [2]:
import pandas as pd
import numpy as np
import math
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading data...")
sales_m   = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861m_sales.csv"))
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
fips_ref  = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"),
                        sep="|", dtype=str)
noaa      = pd.read_csv(os.path.join(OUTPUT_DIR, "noaa_heat_panel.csv"),
                        dtype={"FIPS": str})
acs       = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_poverty_panel.csv"),
                        dtype={"FIPS": str})

# Fix types
sales_m["year"]   = sales_m["year"].astype(int)
sales_m["month"]  = sales_m["month"].astype(int)
territory["year"] = territory["year"].astype(int)

print(f"Monthly sales: {len(sales_m)} rows")
print(f"Territory: {len(territory)} rows")

# ── STEP 1: CLEAN COUNTY NAMES ────────────────────────────────────────────────
def clean_county(name):
    if pd.isna(name):
        return name
    s = str(name)
    s = pd.Series([s]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ")
    s = s.replace(".", "")
    return s.strip()

# Clean FIPS reference
fips_ref.columns    = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"]    = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"]   = fips_ref["state"].str.upper()

# Clean territory
territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

# ── STEP 2: MERGE TERRITORY TO FIPS ──────────────────────────────────────────
territory_fips = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)
print(f"Territory FIPS match rate: "
      f"{territory_fips['FIPS'].notna().mean():.1%}")

# ── STEP 3: MERGE MONTHLY SALES TO TERRITORY ─────────────────────────────────
utility_county_m = sales_m.merge(
    territory_fips[["utility_id", "year", "state", "FIPS", "county_name"]],
    on=["utility_id", "year", "state"], how="inner"
)
print(f"Utility-county-month panel: {len(utility_county_m)} rows")
print(f"Unique FIPS: {utility_county_m['FIPS'].nunique()}")

# ── STEP 4: AGGREGATE TO COUNTY-MONTH ────────────────────────────────────────
county_month = (utility_county_m.dropna(subset=["FIPS"])
                .groupby(["FIPS", "year", "month"])
                .agg(
                    res_revenue_000_total = ("res_revenue_000", "sum"),
                    res_sales_mwh_total   = ("res_sales_mwh",   "sum"),
                    res_customers_total   = ("res_customers",   "sum"),
                    n_utilities           = ("utility_id",      "nunique"),
                )
                .reset_index())

# Implied price
county_month["implied_price_per_mwh"] = np.where(
    (county_month["res_revenue_000_total"] > 0) &
    (county_month["res_sales_mwh_total"]   > 0),
    (county_month["res_revenue_000_total"] * 1000) /
     county_month["res_sales_mwh_total"],
    np.nan
)

# Sales per customer
county_month["res_sales_per_customer"] = np.where(
    county_month["res_customers_total"] > 0,
    county_month["res_sales_mwh_total"] / county_month["res_customers_total"],
    np.nan
)

# Log transforms
county_month["log_res_sales_per_customer"] = np.log(
    county_month["res_sales_per_customer"].replace(0, np.nan))
county_month["log_implied_price"] = np.log(
    county_month["implied_price_per_mwh"].replace(0, np.nan))

# Dominant ownership
ownership_county_m = (utility_county_m
                      .dropna(subset=["FIPS", "res_customers"])
                      .sort_values("res_customers", ascending=False)
                      .groupby(["FIPS", "year", "month"])
                      .agg(dominant_ownership_type=("ownership_type", "first"))
                      .reset_index())

county_month = county_month.merge(
    ownership_county_m, on=["FIPS", "year", "month"], how="left")

print(f"\nCounty-month panel: {len(county_month)} rows")
print(f"Unique counties: {county_month['FIPS'].nunique()}")

# ── STEP 5: MERGE NOAA MONTHLY DATA ──────────────────────────────────────────
noaa["FIPS"] = noaa["FIPS"].str.zfill(5)
acs["FIPS"]  = acs["FIPS"].str.zfill(5)
county_month["FIPS"] = county_month["FIPS"].str.zfill(5)

master_m = county_month.merge(
    noaa[["FIPS", "year", "month",
          "days_above_90f", "days_above_95f", "days_above_100f"]],
    on=["FIPS", "year", "month"], how="left"
)

# ── STEP 6: MERGE ACS POVERTY (ANNUAL) ───────────────────────────────────────
master_m = master_m.merge(
    acs[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
    on=["FIPS", "year"], how="left"
)

# ── STEP 7: CONSTRUCT KEY VARIABLES ──────────────────────────────────────────
mean_pov = master_m["pct_below_150pct_poverty"].mean()
master_m["poverty_demeaned"] = master_m["pct_below_150pct_poverty"] - mean_pov
master_m["heat_x_poverty"]   = (master_m["days_above_95f"] *
                                 master_m["poverty_demeaned"])

# County-month FE identifier
master_m["county_month_fe"] = (master_m["FIPS"] + "_" +
                                master_m["month"].astype(str).str.zfill(2))

print(f"\nMaster monthly panel: {len(master_m)} rows")
print(f"Unique counties: {master_m['FIPS'].nunique()}")
print(f"Years: {sorted(master_m['year'].unique())}")

# Missing values
print(f"\nMissing values:")
for v in ["pct_below_150pct_poverty", "days_above_95f",
          "log_res_sales_per_customer", "log_implied_price"]:
    n = master_m[v].isna().sum()
    print(f"  {v}: {n} ({n/len(master_m):.1%})")

# Summary stats
print(f"\nSummary stats:")
print(master_m[["log_res_sales_per_customer", "log_implied_price",
                 "pct_below_150pct_poverty",
                 "days_above_95f"]].describe().round(3))

# Sanity check: Autauga summer months
autauga = master_m[(master_m["FIPS"] == "01001") &
                   (master_m["year"] == 2020) &
                   (master_m["month"].isin([6, 7, 8]))]
print(f"\nAutauga County 2020 summer months:")
print(autauga[["FIPS", "year", "month", "days_above_95f",
               "log_res_sales_per_customer",
               "pct_below_150pct_poverty"]].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master_m.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly.csv"),
                index=False)
print(f"\nSaved master_panel_monthly.csv")
print(f"Final panel: {len(master_m)} county-month observations")

Loading data...
Monthly sales: 48455 rows
Territory: 118147 rows
Territory FIPS match rate: 99.5%
Utility-county-month panel: 512863 rows
Unique FIPS: 2814

County-month panel: 335878 rows
Unique counties: 2814

Master monthly panel: 335878 rows
Unique counties: 2814
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values:
  pct_below_150pct_poverty: 300 (0.1%)
  days_above_95f: 2412 (0.7%)
  log_res_sales_per_customer: 3 (0.0%)
  log_implied_price: 11 (0.0%)

Summary stats:
       log_res_sales_per_customer  log_implied_price  \
count                  335875.000         335867.000   
mean                       -0.109              4.842   
std                         0.324              0.245   
min                        -6.697              2.560   
25%                        -0.333              4.686   
50%                        -0.111              4.808   
75%                         0.129              4.957   
max                         1.688           

In [11]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"
URDB_PATH  = os.path.join(OUTPUT_DIR, "OpenEI/usurdb.csv.gz")

# ── STEP 1: EXTRACT RESIDENTIAL FIXED CHARGES FROM URDB ──────────────────────
print("Loading URDB...")
cols = ["eiaid", "sector", "is_default", "startdate", "enddate",
        "fixedchargefirstmeter", "fixedchargeunits"]

urdb = pd.read_csv(URDB_PATH, compression="gzip", usecols=cols, low_memory=False)
print(f"Total rates: {len(urdb)}")

# Filter to residential sector only
urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
print(f"Residential rates: {len(urdb_res)}")

# Keep only $/month units (drop $/day)
urdb_res = urdb_res[urdb_res["fixedchargeunits"] == "$/month"].copy()
print(f"Residential $/month rates: {len(urdb_res)}")

# Drop rows with missing fixed charge or missing eiaid
urdb_res = urdb_res.dropna(subset=["fixedchargefirstmeter", "eiaid"])
print(f"After dropping missing: {len(urdb_res)}")

# Convert eiaid to integer for join
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)

# Convert fixed charge to numeric
urdb_res["fixedchargefirstmeter"] = pd.to_numeric(
    urdb_res["fixedchargefirstmeter"], errors="coerce")

# Drop outliers — fixed charges above $100/month are commercial/industrial misclassified
urdb_res = urdb_res[urdb_res["fixedchargefirstmeter"] <= 100]
urdb_res = urdb_res[urdb_res["fixedchargefirstmeter"] >= 0]
print(f"After outlier removal: {len(urdb_res)}")

# Aggregate to utility level — mean fixed charge per utility
# Prefer default rates if available, otherwise use all residential rates
utility_charges = (urdb_res.groupby("eiaid")
                   .agg(
                       fixed_charge_mean = ("fixedchargefirstmeter", "mean"),
                       fixed_charge_median = ("fixedchargefirstmeter", "median"),
                       n_rates = ("fixedchargefirstmeter", "count"),
                       has_default = ("is_default", lambda x: x.eq(True).any())
                   )
                   .reset_index())

print(f"\nUtility-level fixed charges: {len(utility_charges)} utilities")
print(f"Summary stats:")
print(utility_charges["fixed_charge_mean"].describe().round(2))

# ── STEP 2: MERGE TO EIA SALES DATA ──────────────────────────────────────────
# Use annual EIA 861 sales to get utility-to-county mapping with customer counts
sales = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"))
sales["utility_id"] = sales["utility_id"].astype(int)

# Merge fixed charges to sales on utility_id = eiaid
sales_fc = sales.merge(
    utility_charges[["eiaid", "fixed_charge_mean", "fixed_charge_median", "n_rates"]],
    left_on="utility_id", right_on="eiaid", how="left"
)

match_rate = sales_fc["fixed_charge_mean"].notna().mean()
print(f"\nUtility match rate to URDB: {match_rate:.1%}")
print(f"Utilities matched: {sales_fc['fixed_charge_mean'].notna().sum()}")
print(f"Utilities unmatched: {sales_fc['fixed_charge_mean'].isna().sum()}")

# ── STEP 3: AGGREGATE TO COUNTY LEVEL ────────────────────────────────────────
# Load territory file for utility-to-county mapping
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
county_fips = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"),
                          sep="|", dtype=str)

# Clean county names (same function as master merge)
def clean_county(name):
    if pd.isna(name):
        return name
    s = str(name)
    s = pd.Series([s]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ")
    s = s.replace(".", "")
    return s.strip()

county_fips.columns = [c.strip().lower() for c in county_fips.columns]
county_fips["FIPS"] = (county_fips["statefp"].str.zfill(2) +
                        county_fips["countyfp"].str.zfill(3))
county_fips["county_clean"] = county_fips["countyname"].apply(clean_county)
county_fips["state_abbr"]   = county_fips["state"].str.upper()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

territory_fips = territory.merge(
    county_fips[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)

# Merge sales+fixed charges to territory
utility_county_fc = sales_fc.merge(
    territory_fips[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year"], how="inner"
)
utility_county_fc = utility_county_fc.dropna(subset=["FIPS"])

# Customer-weighted average fixed charge per county
# Weight by residential customers so larger utilities count more
def weighted_mean(group, values, weights):
    mask = group[values].notna() & group[weights].notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, values], weights=group.loc[mask, weights])

county_fc = (utility_county_fc.groupby("FIPS")
             .apply(lambda g: pd.Series({
                 "fixed_charge_county": weighted_mean(
                     g, "fixed_charge_mean", "res_customers"),
                 "n_utilities_urdb": g["fixed_charge_mean"].notna().sum(),
                 "pct_utilities_matched": g["fixed_charge_mean"].notna().mean(),
             }))
             .reset_index())

county_fc["FIPS"] = county_fc["FIPS"].str.zfill(5)

print(f"\nCounty-level fixed charges: {len(county_fc)} counties")
print(f"Counties with fixed charge data: {county_fc['fixed_charge_county'].notna().sum()}")
print(f"\nFixed charge distribution:")
print(county_fc["fixed_charge_county"].describe().round(2))

# ── STEP 4: MERGE TO MASTER PANEL ────────────────────────────────────────────
# Fixed charges are time-invariant — merge on FIPS only
master = pd.read_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly.csv"),
                     dtype={"FIPS": str})
master["FIPS"] = master["FIPS"].str.zfill(5)

master_fc = master.merge(
    county_fc[["FIPS", "fixed_charge_county", "pct_utilities_matched"]],
    on="FIPS", how="left"
)

# Demean fixed charge for interpretation
mean_fc = master_fc["fixed_charge_county"].mean()
master_fc["fixed_charge_demeaned"] = master_fc["fixed_charge_county"] - mean_fc

# Triple interaction: heat x poverty x fixed charge
for bin_label in ["bin7_20to25", "bin8_25to30", "bin9_over30"]:
    master_fc[f"{bin_label}_x_pov_x_fc"] = (
        master_fc[bin_label] *
        master_fc["poverty_demeaned"] *
        master_fc["fixed_charge_demeaned"]
    )

print(f"\nMaster panel with fixed charges: {len(master_fc)} rows")
print(f"Missing fixed charge: {master_fc['fixed_charge_county'].isna().sum()} "
      f"({master_fc['fixed_charge_county'].isna().mean():.1%})")
print(f"\nSample — Autauga County:")
print(master_fc[master_fc["FIPS"] == "01001"][
    ["FIPS", "year", "month", "fixed_charge_county",
     "fixed_charge_demeaned"]
].head(3).to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master_fc.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly_fc.csv"),
                 index=False)
county_fc.to_csv(os.path.join(OUTPUT_DIR, "county_fixed_charges.csv"),
                 index=False)

print(f"\nSaved master_panel_monthly_fc.csv")
print(f"Saved county_fixed_charges.csv")
print(f"Mean fixed charge across counties: ${mean_fc:.2f}/month")

Loading URDB...
Total rates: 58275
Residential rates: 13620
Residential $/month rates: 12560
After dropping missing: 12044
After outlier removal: 11972

Utility-level fixed charges: 2451 utilities
Summary stats:
count    2451.00
mean       13.67
std         8.78
min         0.00
25%         7.67
50%        11.50
75%        17.50
max        74.00
Name: fixed_charge_mean, dtype: float64

Utility match rate to URDB: 59.5%
Utilities matched: 17805
Utilities unmatched: 12140


/var/folders/lh/y2t58cg56rx4xfy3f78vhrcw0000gn/T/ipykernel_40479/2217059865.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({



County-level fixed charges: 3115 counties
Counties with fixed charge data: 3104

Fixed charge distribution:
count    3104.00
mean       13.40
std         5.87
min         1.29
25%         9.76
50%        12.26
75%        15.48
max        69.11
Name: fixed_charge_county, dtype: float64

Master panel with fixed charges: 335878 rows
Missing fixed charge: 480 (0.1%)

Sample — Autauga County:
    FIPS  year  month  fixed_charge_county  fixed_charge_demeaned
0  01001  2015      1            10.076729              -2.486643
1  01001  2015      2            10.076729              -2.486643
2  01001  2015      3            10.076729              -2.486643

Saved master_panel_monthly_fc.csv
Saved county_fixed_charges.csv
Mean fixed charge across counties: $12.56/month
